In [1]:
import os
import pandas as pd

In [2]:
DATA_ROOT = "./WESAD"

metadata_txt = {"Subject": [], "Text": []}
metadata_files = sorted([
    os.path.join(root, f)
    for root, dirs, files in os.walk(DATA_ROOT)
    for f in files if f.endswith("readme.txt") and f.startswith("S")
])

print(f"Found {len(metadata_files)} metadata files:")

subjects = [os.path.basename(os.path.dirname(f)) for f in metadata_files]
print(f"Subjects: {subjects}")

for f in metadata_files:
    with open(f, "r") as file:
        content = file.read()
    metadata_txt["Subject"].append(os.path.basename(os.path.dirname(f)))
    metadata_txt["Text"].append(content)

print(f"Metadata entries: {len(metadata_txt['Subject'])}")

print("\nSample metadata entry:")
print("\n'''")
print(f"Subject: {metadata_txt['Subject'][0]}")
print(f"Text: {metadata_txt['Text'][0]}")
print("'''")

Found 15 metadata files:
Subjects: ['S10', 'S11', 'S13', 'S14', 'S15', 'S16', 'S17', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9']
Metadata entries: 15

Sample metadata entry:

'''
Subject: S10
Text: ### Personal information ###
Age: 28
Height (cm): 178
Weight (kg): 76
Gender: male
Dominant hand: right

### Study pre-requisites ###
Did you drink coffee today? NO
Did you drink coffee within the last hour? NO
Did you do any sports today? NO
Are you a smoker? NO
Did you smoke within the last hour? NO
Do you feel ill today? NO

### Additional notes ###
-

'''


In [3]:
# we want to extract the following information from the metadata:

metadata = pd.DataFrame()
text_series = pd.Series(metadata_txt["Text"])

# - Subject ID
metadata["subject_id"] = metadata_txt["Subject"]
# - Age
extract_age = lambda text: int(next((line.split(":")[1].strip() for line in text.splitlines() if "Age" in line), None))
metadata["age"] = text_series.apply(lambda x: extract_age(x))
# - Height
extract_height = lambda text: int(next((line.split(":")[1].strip() for line in text.splitlines() if "Height" in line), None))
metadata["height"] = text_series.apply(lambda x: extract_height(x))
# - Weight
extract_weight = lambda text: int(next((line.split(":")[1].strip() for line in text.splitlines() if "Weight" in line), None))
metadata["weight"] = text_series.apply(lambda x: extract_weight(x))
# - BMI
metadata["bmi"] = round(metadata["weight"] / (metadata["height"] / 100) ** 2, 1)
# - Gender
extract_gender = lambda text: next((line.split(":")[1].strip() for line in text.splitlines() if "Gender" in line), None)
metadata["gender"] = text_series.apply(lambda x: extract_gender(x))
# - Coffee today?
extract_coffee = lambda text: next((line.split("?")[1].strip() for line in text.splitlines() if "coffee today" in line), None)
metadata["coffee_today"] = text_series.apply(lambda x: extract_coffee(x))
# - Coffee last hour?
extract_coffee_last_hour = lambda text: next((line.split("?")[1].strip() for line in text.splitlines() if "coffee within the last hour" in line), None)
metadata["coffee_last_hour"] = text_series.apply(lambda x: extract_coffee_last_hour(x))
# - Sports today?
extract_sports = lambda text: next((line.split("?")[1].strip() for line in text.splitlines() if "sports today" in line), None)
metadata["sports_today"] = text_series.apply(lambda x: extract_sports(x))
# - Smoker?
extract_smoker = lambda text: next((line.split("?")[1].strip() for line in text.splitlines() if "smoker" in line), None)
metadata["smoker"] = text_series.apply(lambda x: extract_smoker(x))
# - Smoke last hour?
extract_smoke_last_hour = lambda text: next((line.split("?")[1].strip() for line in text.splitlines() if "smoke within the last hour" in line), None)
metadata["smoke_last_hour"] = text_series.apply(lambda x: extract_smoke_last_hour(x))
# - Ill today?
extract_ill = lambda text: next((line.split("?")[1].strip() for line in text.splitlines() if "ill today" in line), None)
metadata["ill_today"] = text_series.apply(lambda x: extract_ill(x))

print("\nExtracted metadata:")
display(metadata)



Extracted metadata:


,subject_id,age,height,weight,bmi,gender,coffee_today,coffee_last_hour,sports_today,smoker,smoke_last_hour,ill_today
0,S10,28,178,76,24.0,male,NO,NO,NO,NO,NO,NO
1,S11,26,171,54,18.5,female,YES,NO,NO,NO,NO,NO
2,S13,28,181,82,25.0,male,NO,NO,NO,NO,NO,NO
3,S14,27,180,80,24.7,male,NO,NO,NO,NO,NO,NO
4,S15,28,186,83,24.0,male,NO,NO,NO,NO,NO,NO
5,S16,24,184,69,20.4,male,NO,NO,NO,NO,NO,NO
6,S17,29,165,55,20.2,female,NO,NO,NO,NO,NO,NO
7,S2,27,175,80,26.1,male,NO,NO,NO,NO,NO,NO
8,S3,27,173,69,23.1,male,NO,NO,NO,NO,NO,NO
9,S4,25,175,90,29.4,male,NO,NO,NO,NO,NO,NO


In [6]:
# one hot encode the categorical variables except subject_id
metadata_encoded = pd.get_dummies(metadata.drop(columns=["subject_id"]), drop_first=True, dtype=int)
metadata_encoded["subject_id"] = metadata["subject_id"]
cols = ["subject_id"] + [col for col in metadata_encoded.columns if col != "subject_id"]
metadata_encoded = metadata_encoded[cols]

print("\nOne-hot encoded metadata (Numeric):")
display(metadata_encoded)


One-hot encoded metadata (Numeric):


,subject_id,age,height,weight,bmi,gender_male,coffee_today_YES,sports_today_YES,smoker_YES,ill_today_YES
0,S10,28,178,76,24.0,1,0,0,0,0
1,S11,26,171,54,18.5,0,1,0,0,0
2,S13,28,181,82,25.0,1,0,0,0,0
3,S14,27,180,80,24.7,1,0,0,0,0
4,S15,28,186,83,24.0,1,0,0,0,0
5,S16,24,184,69,20.4,1,0,0,0,0
6,S17,29,165,55,20.2,0,0,0,0,0
7,S2,27,175,80,26.1,1,0,0,0,0
8,S3,27,173,69,23.1,1,0,0,0,0
9,S4,25,175,90,29.4,1,0,0,0,0
